# TriML — Full ML Pipeline (Colab Edition)
**CS 6140 Machine Learning — Northeastern University**

This notebook runs the complete TriML pipeline:
1. Download data from Zenodo
2. Load & merge 3 CSVs (athletes, daily_data, activity_data)
3. Engineer features (ACWR, HRV z-score, sleep composite, RHR trend, Grit Score)
4. Train 9 models with 5-fold GroupKFold CV
5. Hyperparameter sweep (6 models)
6. Generate 10 publication-quality plots

**Runtime:** Select `Runtime > Change runtime type > T4 GPU` for best performance (~10 min total).

## 0. Install Dependencies

In [ ]:
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn scipy imbalanced-learn

## 1. Download Data from Zenodo

In [ ]:
import urllib.request
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

ZENODO_URLS = {
    "athletes.csv": "https://zenodo.org/api/records/15401061/files/athletes.csv/content",
    "daily_data.csv": "https://zenodo.org/api/records/15401061/files/daily_data.csv/content",
    "activity_data.csv": "https://zenodo.org/api/records/15401061/files/activity_data.csv/content",
}

for fname, url in ZENODO_URLS.items():
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest):
        print(f"  {fname} already exists, skipping")
        continue
    print(f"  Downloading {fname}...", end=" ", flush=True)
    urllib.request.urlretrieve(url, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"done ({size_mb:.1f} MB)")

print("\nAll data files ready!")
!ls -lh /content/data/

## 2. Data Loading & Parsing

In [ ]:
import ast
import re
import time
import pickle
import warnings
from pathlib import Path
from typing import Optional, Union

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, mean_squared_error,
    precision_score, r2_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

device_info = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device_info}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ========================================================================
# LOADER — Parse all 3 CSVs
# ========================================================================

def _parse_hrv_range(s):
    nums = re.findall(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", str(s))
    return float(nums[0]), float(nums[1])

def _parse_hr_zones_athlete(s):
    cleaned = re.sub(r"np\.float64\(([^)]+)\)", r"\1", str(s))
    return ast.literal_eval(cleaned)

def _parse_zone_dict(s):
    if pd.isna(s):
        return None
    return ast.literal_eval(str(s))

def load_athletes(path):
    df = pd.read_csv(path)
    parsed = df["hrv_range"].apply(_parse_hrv_range)
    df["hrv_min"] = parsed.apply(lambda t: t[0])
    df["hrv_max"] = parsed.apply(lambda t: t[1])
    df.drop(columns=["hrv_range"], inplace=True)
    parsed_zones = df["hr_zones"].apply(_parse_hr_zones_athlete)
    for z in range(1, 7):
        key = f"Z{z}"
        df[f"hr_zone{z}_lo"] = parsed_zones.apply(lambda d, k=key: float(d[k][0]))
        df[f"hr_zone{z}_hi"] = parsed_zones.apply(lambda d, k=key: float(d[k][1]))
    df.drop(columns=["hr_zones"], inplace=True)
    return df

def load_daily(path):
    df = pd.read_csv(path, parse_dates=["date"])
    df.sort_values(["athlete_id", "date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

def load_activities(path):
    df = pd.read_csv(path, parse_dates=["date"])
    df.sort_values(["athlete_id", "date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    parsed_hr = df["hr_zones"].apply(_parse_zone_dict)
    for z in range(1, 7):
        key = f"Z{z}"
        df[f"hr_z{z}_pct"] = parsed_hr.apply(lambda d, k=key: float(d[k]) if d is not None else 0.0)
    df.drop(columns=["hr_zones"], inplace=True)
    parsed_pwr = df["power_zones"].apply(_parse_zone_dict)
    for z in range(1, 8):
        key = f"Z{z}"
        df[f"pwr_z{z}_pct"] = parsed_pwr.apply(lambda d, k=key: float(d[k]) if d is not None else 0.0)
    df.drop(columns=["power_zones"], inplace=True)
    return df

def aggregate_activities(df_act):
    zone_cols = [f"hr_z{z}_pct" for z in range(1, 7)] + [f"pwr_z{z}_pct" for z in range(1, 8)]
    sport_dummies = pd.get_dummies(df_act["sport"], prefix="n")
    df_act = pd.concat([df_act, sport_dummies], axis=1)
    sport_count_cols = [c for c in df_act.columns if c.startswith("n_")]

    def _dominant_sport(sub):
        return sub.groupby("sport")["tss"].sum().idxmax()

    dominant = (
        df_act.groupby(["athlete_id", "date"])
        .apply(_dominant_sport, include_groups=False)
        .rename("dominant_sport")
        .reset_index()
    )
    sports_list = (
        df_act.groupby(["athlete_id", "date"])["sport"]
        .apply(lambda x: ",".join(sorted(x.unique())))
        .rename("sports_of_day")
        .reset_index()
    )
    agg_dict = {
        "tss": "sum", "duration_minutes": "sum", "work_kilojoules": "sum",
        "intensity_factor": "mean", "avg_hr": "mean",
        "training_effect_aerobic": "mean", "training_effect_anaerobic": "mean",
    }
    for col in zone_cols:
        agg_dict[col] = "mean"
    for col in sport_count_cols:
        agg_dict[col] = "sum"
    agg = df_act.groupby(["athlete_id", "date"]).agg(agg_dict).reset_index()
    for sport in ["bike", "run", "swim", "strength"]:
        old = f"n_{sport}"
        if old not in agg.columns:
            agg[old] = 0
    agg = agg.merge(dominant, on=["athlete_id", "date"], how="left")
    agg = agg.merge(sports_list, on=["athlete_id", "date"], how="left")
    return agg

def build_merged(df_daily, df_act_agg, df_athletes):
    df = df_daily.merge(df_act_agg, on=["athlete_id", "date"], how="left")
    activity_numeric_cols = [
        c for c in df_act_agg.columns
        if c not in ("athlete_id", "date", "dominant_sport", "sports_of_day")
    ]
    df[activity_numeric_cols] = df[activity_numeric_cols].fillna(0)
    df["dominant_sport"] = df["dominant_sport"].fillna("rest")
    df["sports_of_day"] = df["sports_of_day"].fillna("")
    df_athletes = df_athletes.rename(columns={
        "resting_hr": "baseline_rhr",
        "sleep_quality": "baseline_sleep_quality",
    })
    df = df.merge(df_athletes, on="athlete_id", how="left")
    df.sort_values(["athlete_id", "date"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

print("Loader functions defined.")

In [ ]:
# Load all 3 CSVs
t0 = time.time()

print("Loading athletes.csv...", flush=True)
ath = load_athletes(f"{DATA_DIR}/athletes.csv")
print(f"  {ath.shape}")

print("Loading daily_data.csv...", flush=True)
daily = load_daily(f"{DATA_DIR}/daily_data.csv")
print(f"  {daily.shape}")

print("Loading activity_data.csv...", flush=True)
act = load_activities(f"{DATA_DIR}/activity_data.csv")
print(f"  {act.shape}")

print("\nAggregating activities by day...", flush=True)
act_agg = aggregate_activities(act)

print("Merging all tables...", flush=True)
merged = build_merged(daily, act_agg, ath)
print(f"\nMerged shape: {merged.shape}  ({time.time()-t0:.1f}s)")

## 3. Feature Engineering

In [ ]:
# ========================================================================
# FEATURES — ACWR, HRV z-score, sleep composite, RHR trend, Grit Score
# ========================================================================

MIN_HISTORY_DAYS = 28
ACWR_HIGH = 1.3
ACWR_LOW  = 0.8
LOAD_CLASSES = ["Undertrained", "Balanced", "Overreaching"]

FEATURE_COLS = [
    "acwr", "hrv_zscore", "sleep_composite_z", "rhr_trend",
    "body_battery_morning", "stress", "sleep_hours", "deep_sleep",
    "rem_sleep", "sleep_quality", "hrv", "resting_hr",
    "tss", "duration_minutes", "intensity_factor",
    "training_effect_aerobic", "training_effect_anaerobic",
    "age", "vo2max", "ftp", "training_experience",
    "weekly_training_hours", "gender_enc", "lifestyle_enc",
]

def _rolling_slope(series, window=7):
    slopes = np.full(len(series), np.nan)
    vals = series.values
    for i in range(len(vals)):
        start = max(0, i - window + 1)
        chunk = vals[start : i + 1]
        chunk = chunk[~np.isnan(chunk)]
        if len(chunk) < 3:
            continue
        x = np.arange(len(chunk), dtype=float)
        xm = x - x.mean()
        slopes[i] = np.dot(xm, chunk - chunk.mean()) / np.dot(xm, xm)
    return pd.Series(slopes, index=series.index)

def _engineer_athlete(df):
    df = df.copy()
    acute   = df["tss"].rolling(7,  min_periods=1).mean()
    chronic = df["tss"].rolling(28, min_periods=7).mean()
    df["acwr"] = (acute / chronic.replace(0, np.nan)).clip(0, 3)

    hrv_14m = df["hrv"].rolling(14, min_periods=5).mean()
    hrv_14s = df["hrv"].rolling(14, min_periods=5).std().replace(0, np.nan)
    df["hrv_zscore"] = (df["hrv"] - hrv_14m) / hrv_14s

    raw_sleep = df["sleep_hours"] * df["sleep_quality"]
    mu = raw_sleep.mean()
    sd = raw_sleep.std()
    df["sleep_composite_z"] = (raw_sleep - mu) / (sd if sd > 0 else 1)

    df["rhr_trend"] = _rolling_slope(df["resting_hr"], window=7)

    # Grit Score: HIGH = dangerous/overreaching, LOW = safe/undertrained
    hrv_sub   = 1 / (1 + np.exp(df["hrv_zscore"]))
    sleep_sub = 1 / (1 + np.exp(df["sleep_composite_z"]))
    bat_sub   = 1 - (df["body_battery_morning"] / 100).clip(0, 1)
    stress_max = df["stress"].max() if df["stress"].max() > 0 else 1
    stress_sub = (df["stress"] / stress_max).clip(0, 1)
    acwr_sub  = (df["acwr"] - 1.0).abs().clip(0, 1)

    df["grit_score"] = 100 * (
        0.25 * hrv_sub + 0.25 * sleep_sub + 0.20 * bat_sub
        + 0.15 * stress_sub + 0.15 * acwr_sub
    )
    return df

def engineer_features(df_merged):
    gender_enc    = LabelEncoder().fit(df_merged["gender"])
    lifestyle_enc = LabelEncoder().fit(df_merged["lifestyle"])
    df_merged = df_merged.copy()
    df_merged["gender_enc"]    = gender_enc.transform(df_merged["gender"])
    df_merged["lifestyle_enc"] = lifestyle_enc.transform(df_merged["lifestyle"])

    parts = []
    for _, grp in df_merged.groupby("athlete_id", sort=False):
        parts.append(_engineer_athlete(grp.sort_values("date")))

    out = pd.concat(parts).sort_values(["athlete_id", "date"]).reset_index(drop=True)

    q_low  = out["grit_score"].quantile(0.25)
    q_high = out["grit_score"].quantile(0.75)
    def _grit_class(v):
        if pd.isna(v): return 1
        if v >= q_high: return 2  # Overreaching
        if v <= q_low:  return 0  # Undertrained
        return 1                  # Balanced

    out["load_class"] = out["grit_score"].apply(_grit_class)
    out["grit_q_low"]  = q_low
    out["grit_q_high"] = q_high
    return out

def get_feature_matrix(df_feat):
    def _drop_head(grp):
        return grp.iloc[MIN_HISTORY_DAYS:]
    df = df_feat.groupby("athlete_id", group_keys=False).apply(_drop_head)
    needed = FEATURE_COLS + ["injury", "grit_score", "load_class", "athlete_id"]
    df = df[needed].dropna().reset_index(drop=True)
    X       = df[FEATURE_COLS].values.astype(np.float32)
    y_class = df["injury"].values.astype(np.int64)
    y_grit  = df["grit_score"].values.astype(np.float32)
    y_load  = df["load_class"].values.astype(np.int64)
    groups  = df["athlete_id"].values
    return X, y_class, y_grit, y_load, groups, FEATURE_COLS

print("Feature engineering functions defined.")

In [ ]:
# Run feature engineering
t1 = time.time()
print("Engineering features (per-athlete rolling windows)...", flush=True)
df_feat = engineer_features(merged)

for col in ("acwr", "hrv_zscore", "sleep_composite_z", "rhr_trend", "grit_score"):
    s = df_feat[col].dropna()
    print(f"  {col:<22} mean={s.mean():7.3f}  std={s.std():6.3f}  min={s.min():7.3f}  max={s.max():7.3f}")

load_dist = df_feat["load_class"].value_counts().sort_index()
print(f"\nLoad class distribution:")
for cls_id, count in load_dist.items():
    pct = 100 * count / len(df_feat)
    print(f"  {LOAD_CLASSES[cls_id]:<15} {count:>7,}  ({pct:.1f}%)")

print(f"\nInjury prevalence: {100*df_feat['injury'].mean():.2f}%")
print(f"Feature engineering done ({time.time()-t1:.1f}s)")

In [ ]:
# Build feature matrix
X, y_injury, y_grit, y_load, groups, feat_names = get_feature_matrix(df_feat)

print(f"X shape      : {X.shape}")
print(f"y_injury     : {np.bincount(y_injury)}  (0=healthy, 1=injured)")
print(f"y_load       : {np.bincount(y_load)}  (0=Undertrained, 1=Balanced, 2=Overreaching)")
print(f"grit_score   : mean={y_grit.mean():.1f}  std={y_grit.std():.1f}")
print(f"Unique groups: {len(np.unique(groups))} athletes")

## 4. Model Definitions (PyTorch MLP + sklearn)

In [ ]:
# ========================================================================
# MODELS — MLP, CV runners, HP sweep
# ========================================================================

N_FOLDS = 5
RANDOM_STATE = 42

class MLP(nn.Module):
    """3-hidden-layer MLP with BatchNorm and Dropout."""
    def __init__(self, in_features, out_features, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden // 2, hidden // 4),
            nn.BatchNorm1d(hidden // 4),
            nn.ReLU(),
            nn.Linear(hidden // 4, out_features),
        )
    def forward(self, x):
        return self.net(x)


def _train_mlp(X_train, y_train, X_val, task, n_classes=1,
               hidden=128, epochs=30, batch_size=1024, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train).astype(np.float32)
    X_vl = scaler.transform(X_val).astype(np.float32)

    out_dim = 1 if task in ("binary", "regression") else n_classes
    model = MLP(X_tr.shape[1], out_dim, hidden=hidden).to(device)

    if task == "binary":
        pos_weight = torch.tensor(
            [(y_train == 0).sum() / max((y_train == 1).sum(), 1)],
            dtype=torch.float32,
        ).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    elif task == "multiclass":
        criterion = nn.CrossEntropyLoss()
    else:
        criterion = nn.MSELoss()

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    Xt = torch.from_numpy(X_tr)
    if task in ("binary", "regression"):
        yt = torch.from_numpy(y_train.astype(np.float32)).unsqueeze(1)
    else:
        yt = torch.from_numpy(y_train.astype(np.int64))

    ds = TensorDataset(Xt, yt)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            opt.step()
        scheduler.step()

    model.eval()
    return model, scaler, X_vl, device


def _predict_mlp(model, X_val_scaled, device, task, n_classes=1):
    with torch.no_grad():
        Xv = torch.from_numpy(X_val_scaled).to(device)
        logits = model(Xv).cpu().numpy()
    if task == "binary":
        proba = 1 / (1 + np.exp(-logits.squeeze()))
        pred  = (proba >= 0.5).astype(int)
        return pred, proba
    elif task == "multiclass":
        from scipy.special import softmax
        proba = softmax(logits, axis=1)
        pred  = proba.argmax(axis=1)
        return pred, proba
    else:
        return logits.squeeze(), None


def _clf_metrics(y_true, y_pred, y_proba, n_classes):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    try:
        if n_classes == 2:
            auc = roc_auc_score(y_true, y_proba)
        else:
            auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except Exception:
        auc = np.nan
    return {"accuracy": acc, "f1_macro": f1, "precision": prec, "recall": rec, "roc_auc": auc}


def _reg_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mae": mae, "r2": r2}

print("Model functions defined.")

## 5. Train All 9 Models (5-fold GroupKFold CV)

In [ ]:
def run_classification_cv(X, y, groups, n_classes=2, label="injury"):
    gkf = GroupKFold(n_splits=N_FOLDS)
    task = "binary" if n_classes == 2 else "multiclass"
    results = {name: {"fold_metrics": []} for name in ("lr", "rf", "mlp")}
    rf_importances = []
    scaler_global = StandardScaler()
    X_scaled = scaler_global.fit_transform(X)

    for fold, (tr_idx, vl_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_vl = X_scaled[tr_idx], X_scaled[vl_idx]
        y_tr, y_vl = y[tr_idx], y[vl_idx]
        print(f"  {label} fold {fold+1}/{N_FOLDS}...", flush=True)

        # LR
        lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, C=0.5)
        lr.fit(X_tr, y_tr)
        y_pred_lr = lr.predict(X_vl)
        y_prob_lr = lr.predict_proba(X_vl) if n_classes > 2 else lr.predict_proba(X_vl)[:, 1]
        results["lr"]["fold_metrics"].append(_clf_metrics(y_vl, y_pred_lr, y_prob_lr, n_classes))

        # RF
        rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight="balanced",
                                     random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        y_pred_rf = rf.predict(X_vl)
        y_prob_rf = rf.predict_proba(X_vl) if n_classes > 2 else rf.predict_proba(X_vl)[:, 1]
        results["rf"]["fold_metrics"].append(_clf_metrics(y_vl, y_pred_rf, y_prob_rf, n_classes))
        rf_importances.append(rf.feature_importances_)

        # MLP
        model, _, X_vl_sc, device = _train_mlp(X[tr_idx], y_tr, X[vl_idx], task=task, n_classes=n_classes)
        y_pred_mlp, y_prob_mlp = _predict_mlp(model, X_vl_sc, device, task, n_classes)
        results["mlp"]["fold_metrics"].append(_clf_metrics(y_vl, y_pred_mlp, y_prob_mlp, n_classes))

    for name in ("lr", "rf", "mlp"):
        folds = results[name]["fold_metrics"]
        keys  = folds[0].keys()
        results[name]["mean"] = {k: np.mean([f[k] for f in folds]) for k in keys}
        results[name]["std"]  = {k: np.std([f[k] for f in folds]) for k in keys}

    results["feature_importance"] = np.mean(rf_importances, axis=0)
    return results


def run_regression_cv(X, y, groups, label="grit_score"):
    gkf = GroupKFold(n_splits=N_FOLDS)
    results = {name: {"fold_metrics": []} for name in ("lasso", "rf", "mlp")}
    rf_importances = []

    for fold, (tr_idx, vl_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_vl = X[tr_idx], X[vl_idx]
        y_tr, y_vl = y[tr_idx], y[vl_idx]
        print(f"  {label} fold {fold+1}/{N_FOLDS}...", flush=True)

        # Lasso + Poly
        lasso_pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
            ("lasso", Lasso(alpha=0.01, max_iter=5000)),
        ])
        lasso_pipe.fit(X_tr, y_tr)
        results["lasso"]["fold_metrics"].append(_reg_metrics(y_vl, lasso_pipe.predict(X_vl)))

        # RF
        rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        results["rf"]["fold_metrics"].append(_reg_metrics(y_vl, rf.predict(X_vl)))
        rf_importances.append(rf.feature_importances_)

        # MLP
        model, _, X_vl_sc, device = _train_mlp(X_tr, y_tr, X_vl, task="regression")
        y_pred_mlp, _ = _predict_mlp(model, X_vl_sc, device, "regression")
        results["mlp"]["fold_metrics"].append(_reg_metrics(y_vl, y_pred_mlp))

    for name in ("lasso", "rf", "mlp"):
        folds = results[name]["fold_metrics"]
        keys  = folds[0].keys()
        results[name]["mean"] = {k: np.mean([f[k] for f in folds]) for k in keys}
        results[name]["std"]  = {k: np.std([f[k] for f in folds]) for k in keys}

    results["feature_importance"] = np.mean(rf_importances, axis=0)
    return results

print("CV runner functions defined.")

In [ ]:
%%time
# ==================== RUN ALL 9 MODELS ====================
print("="*60)
print("  Training 9 models x 5 folds = 45 training runs")
print("  (GPU accelerated — should take ~5 minutes)")
print("="*60)

print("\n--- Injury Classification (binary) ---")
injury_clf = run_classification_cv(X, y_injury, groups, n_classes=2, label="injury")

print("\n--- Load Class Classification (3-class) ---")
load_clf = run_classification_cv(X, y_load, groups, n_classes=3, label="load_class")

print("\n--- Grit Score Regression ---")
grit_reg = run_regression_cv(X, y_grit, groups, label="grit_score")

raw_results = {
    "injury_clf": injury_clf,
    "load_clf":   load_clf,
    "grit_reg":   grit_reg,
    "feature_names": feat_names,
}

print("\n" + "="*60)
print("  ALL MODELS COMPLETE!")
print("="*60)

In [ ]:
# ==================== PRINT RESULTS TABLES ====================

def print_clf_table(res, label):
    print(f"\n{'='*70}")
    print(f"  Classification — {label}")
    print(f"{'='*70}")
    model_keys = [("lr", "Logistic Regression"), ("rf", "Random Forest"), ("mlp", "DNN (MLP)")]
    header = f"{'Model':<22} {'ROC-AUC':>12} {'F1-macro':>10} {'Precision':>11} {'Recall':>9} {'Accuracy':>10}"
    print(header)
    print("-" * len(header))
    for key, name in model_keys:
        m = res[key]["mean"]
        s = res[key]["std"]
        print(f"{name:<22} "
              f"{m['roc_auc']:>6.3f}\u00b1{s['roc_auc']:.3f}  "
              f"{m['f1_macro']:>5.3f}\u00b1{s['f1_macro']:.3f}  "
              f"{m['precision']:>5.3f}\u00b1{s['precision']:.3f}  "
              f"{m['recall']:>5.3f}\u00b1{s['recall']:.3f}  "
              f"{m['accuracy']:>5.3f}\u00b1{s['accuracy']:.3f}")

def print_reg_table(res, label):
    print(f"\n{'='*70}")
    print(f"  Regression — {label}")
    print(f"{'='*70}")
    model_keys = [("lasso", "Lasso + Poly"), ("rf", "Random Forest"), ("mlp", "DNN (MLP)")]
    header = f"{'Model':<22} {'RMSE':>12} {'MAE':>12} {'R\u00b2':>12}"
    print(header)
    print("-" * len(header))
    for key, name in model_keys:
        m = res[key]["mean"]
        s = res[key]["std"]
        print(f"{name:<22} {m['rmse']:>6.3f}\u00b1{s['rmse']:.3f}  {m['mae']:>6.3f}\u00b1{s['mae']:.3f}  {m['r2']:>6.3f}\u00b1{s['r2']:.3f}")

def print_feature_importances(fi, feature_names, label, top_n=10):
    print(f"\n{'='*70}")
    print(f"  Top {top_n} Feature Importances (RF) — {label}")
    print(f"{'='*70}")
    ranked = sorted(zip(feature_names, fi), key=lambda x: x[1], reverse=True)
    for i, (fname, imp) in enumerate(ranked[:top_n], 1):
        bar = "\u2588" * int(imp * 200)
        print(f"  {i:2}. {fname:<30} {imp:.4f}  {bar}")

print_clf_table(raw_results["injury_clf"], "Injury (binary)")
print_clf_table(raw_results["load_clf"],   "Load Class (3-class)")
print_reg_table(raw_results["grit_reg"],   "Grit Score (0-100)")
print_feature_importances(raw_results["injury_clf"]["feature_importance"], feat_names, "Injury")
print_feature_importances(raw_results["grit_reg"]["feature_importance"], feat_names, "Grit Score")

## 6. Hyperparameter Sweep

In [ ]:
%%time
# ==================== HP SWEEP ====================
print("="*60)
print("  Hyperparameter Sweep (3-fold GroupKFold)")
print("  Sweeping: LR C, RF max_depth, DNN hidden_size")
print("  for both classification (load_class) and regression (grit_score)")
print("="*60)

gkf3 = GroupKFold(n_splits=3)
scaler_hp = StandardScaler()
X_sc = scaler_hp.fit_transform(X)
n_classes_hp = 3
task_clf_hp = "multiclass"

def _cv_score_clf(model_fn):
    scores = []
    for tr, vl in gkf3.split(X, y_load, groups):
        m = model_fn()
        m.fit(X_sc[tr], y_load[tr])
        proba = m.predict_proba(X_sc[vl])
        try:
            s = roc_auc_score(y_load[vl], proba, multi_class="ovr", average="macro")
        except Exception:
            s = np.nan
        scores.append(s)
    return np.nanmean(scores), np.nanstd(scores)

def _cv_score_reg(model_fn):
    scores = []
    for tr, vl in gkf3.split(X, y_grit, groups):
        m = model_fn()
        m.fit(X[tr], y_grit[tr])
        scores.append(r2_score(y_grit[vl], m.predict(X[vl])))
    return np.nanmean(scores), np.nanstd(scores)

def _cv_score_lasso(alpha):
    scores = []
    for tr, vl in gkf3.split(X, y_grit, groups):
        pipe = Pipeline([
            ("sc", StandardScaler()),
            ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
            ("lasso", Lasso(alpha=alpha, max_iter=5000)),
        ])
        pipe.fit(X[tr], y_grit[tr])
        scores.append(r2_score(y_grit[vl], pipe.predict(X[vl])))
    return np.nanmean(scores), np.nanstd(scores)

def _cv_score_mlp_clf(hidden):
    scores = []
    for tr, vl in gkf3.split(X, y_load, groups):
        model, _, X_vl_sc, device = _train_mlp(
            X[tr], y_load[tr], X[vl], task=task_clf_hp,
            n_classes=n_classes_hp, hidden=hidden, epochs=20,
        )
        _, proba = _predict_mlp(model, X_vl_sc, device, task_clf_hp, n_classes_hp)
        try:
            s = roc_auc_score(y_load[vl], proba, multi_class="ovr", average="macro")
        except Exception:
            s = np.nan
        scores.append(s)
    return np.nanmean(scores), np.nanstd(scores)

def _cv_score_mlp_reg(hidden):
    scores = []
    for tr, vl in gkf3.split(X, y_grit, groups):
        model, _, X_vl_sc, device = _train_mlp(
            X[tr], y_grit[tr], X[vl], task="regression",
            hidden=hidden, epochs=20,
        )
        pred, _ = _predict_mlp(model, X_vl_sc, device, "regression")
        scores.append(r2_score(y_grit[vl], pred))
    return np.nanmean(scores), np.nanstd(scores)

hp_results = {}

# --- LR: C ---
print("\n  Sweeping LR C...", flush=True)
C_vals = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
rows = []
for c in C_vals:
    mu, sd = _cv_score_clf(
        lambda c=c: LogisticRegression(C=c, max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    )
    rows.append({"C": c, "mean_auc": mu, "std_auc": sd})
    print(f"    C={c:<8} AUC={mu:.4f} +/- {sd:.4f}")
hp_results["lr_C"] = pd.DataFrame(rows)

# --- RF clf: max_depth ---
print("\n  Sweeping RF classifier max_depth...", flush=True)
depths = [3, 5, 8, 12, 16, 20]
rows = []
for d in depths:
    mu, sd = _cv_score_clf(
        lambda d=d: RandomForestClassifier(n_estimators=100, max_depth=d, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1)
    )
    rows.append({"max_depth": d, "mean_auc": mu, "std_auc": sd})
    print(f"    depth={d:<5} AUC={mu:.4f} +/- {sd:.4f}")
hp_results["rf_clf_depth"] = pd.DataFrame(rows)

# --- DNN clf: hidden size ---
print("\n  Sweeping DNN classifier hidden size...", flush=True)
hidden_sizes = [32, 64, 128, 256]
rows = []
for h in hidden_sizes:
    mu, sd = _cv_score_mlp_clf(h)
    rows.append({"hidden_size": h, "mean_auc": mu, "std_auc": sd})
    print(f"    hidden={h:<5} AUC={mu:.4f} +/- {sd:.4f}")
hp_results["dnn_clf_hidden"] = pd.DataFrame(rows)

# --- Lasso: alpha ---
print("\n  Sweeping Lasso alpha...", flush=True)
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0]
rows = []
for a in alphas:
    mu, sd = _cv_score_lasso(a)
    rows.append({"alpha": a, "mean_r2": mu, "std_r2": sd})
    print(f"    alpha={a:<8} R2={mu:.4f} +/- {sd:.4f}")
hp_results["lasso_alpha"] = pd.DataFrame(rows)

# --- RF reg: max_depth ---
print("\n  Sweeping RF regressor max_depth...", flush=True)
rows = []
for d in depths:
    mu, sd = _cv_score_reg(
        lambda d=d: RandomForestRegressor(n_estimators=100, max_depth=d, random_state=RANDOM_STATE, n_jobs=-1)
    )
    rows.append({"max_depth": d, "mean_r2": mu, "std_r2": sd})
    print(f"    depth={d:<5} R2={mu:.4f} +/- {sd:.4f}")
hp_results["rf_reg_depth"] = pd.DataFrame(rows)

# --- DNN reg: hidden size ---
print("\n  Sweeping DNN regressor hidden size...", flush=True)
rows = []
for h in hidden_sizes:
    mu, sd = _cv_score_mlp_reg(h)
    rows.append({"hidden_size": h, "mean_r2": mu, "std_r2": sd})
    print(f"    hidden={h:<5} R2={mu:.4f} +/- {sd:.4f}")
hp_results["dnn_reg_hidden"] = pd.DataFrame(rows)

print("\n" + "="*60)
print("  HP SWEEP COMPLETE!")
print("="*60)

In [ ]:
# Print HP sweep summary tables
sweep_info = [
    ("lr_C",           "LR — C (regularization)",     "C",           "mean_auc", "std_auc",  "ROC-AUC"),
    ("rf_clf_depth",   "RF Classifier — max_depth",   "max_depth",   "mean_auc", "std_auc",  "ROC-AUC"),
    ("dnn_clf_hidden", "DNN Classifier — hidden size", "hidden_size", "mean_auc", "std_auc",  "ROC-AUC"),
    ("lasso_alpha",    "Lasso — alpha",                "alpha",       "mean_r2",  "std_r2",   "R\u00b2"),
    ("rf_reg_depth",   "RF Regressor — max_depth",    "max_depth",   "mean_r2",  "std_r2",   "R\u00b2"),
    ("dnn_reg_hidden", "DNN Regressor — hidden size",  "hidden_size", "mean_r2",  "std_r2",   "R\u00b2"),
]

for key, title, hp_col, metric_col, std_col, metric_name in sweep_info:
    df = hp_results[key]
    print(f"\n{'='*50}")
    print(f"  {title}")
    print(f"{'='*50}")
    print(f"  {hp_col:<15} {metric_name:>8}   std")
    print(f"  {'\u2500'*35}")
    for _, row in df.iterrows():
        best = row[metric_col] == df[metric_col].max()
        flag = " << BEST" if best else ""
        print(f"  {str(row[hp_col]):<15} {row[metric_col]:>8.4f}  \u00b1{row[std_col]:.4f}{flag}")

## 7. Generate All 10 Plots

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from math import pi

PLOTS_DIR = "/content/plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

# Style
plt.style.use("dark_background")

DARK_BG   = "#0d1117"
PANEL_BG  = "#161b22"
GRID_COL  = "#30363d"
TEXT_COL  = "#e6edf3"
ACCENT1   = "#4A90D9"   # blue
ACCENT2   = "#D94040"   # red
ACCENT3   = "#F5C842"   # yellow
ACCENT4   = "#FFFFFF"   # white
ACCENT5   = "#4A90D9"   # blue alias

MODEL_COLORS = {"LR": ACCENT1, "RF": ACCENT2, "DNN": ACCENT3, "Lasso": ACCENT4}

RCPARAMS = {
    "figure.facecolor": DARK_BG, "axes.facecolor": PANEL_BG,
    "axes.edgecolor": GRID_COL, "axes.labelcolor": TEXT_COL,
    "axes.titlecolor": TEXT_COL, "xtick.color": TEXT_COL,
    "ytick.color": TEXT_COL, "text.color": TEXT_COL,
    "grid.color": GRID_COL, "grid.linestyle": "--", "grid.alpha": 0.5,
    "legend.facecolor": PANEL_BG, "legend.edgecolor": GRID_COL,
    "font.size": 11, "axes.titlesize": 16, "axes.labelsize": 12,
}
plt.rcParams.update(RCPARAMS)

# Tables for feature importances
def _fi_df(res, feature_names, top_n=15):
    fi = pd.Series(res["feature_importance"], index=feature_names)
    return fi.nlargest(top_n).reset_index().rename(columns={"index": "Feature", 0: "Importance"})

tables = {
    "fi_injury": _fi_df(raw_results["injury_clf"], feat_names),
    "fi_grit":   _fi_df(raw_results["grit_reg"], feat_names),
}

df_sample = df_feat.sample(min(5000, len(df_feat)), random_state=42)

FEATURE_CATEGORIES = {
    "deep_sleep": ("sleep", "#4A90D9"), "sleep_quality": ("sleep", "#4A90D9"),
    "sleep_hours": ("sleep", "#4A90D9"), "rem_sleep": ("sleep", "#4A90D9"),
    "sleep_composite_z": ("sleep", "#4A90D9"),
    "hrv_zscore": ("hrv", "#FFFFFF"), "hrv": ("hrv", "#FFFFFF"),
    "rhr_trend": ("hrv", "#FFFFFF"), "resting_hr": ("hrv", "#FFFFFF"),
    "acwr": ("training", "#F5C842"), "tss": ("training", "#F5C842"),
    "duration_minutes": ("training", "#F5C842"), "intensity_factor": ("training", "#F5C842"),
    "training_effect_aerobic": ("training", "#F5C842"), "training_effect_anaerobic": ("training", "#F5C842"),
    "weekly_training_hours": ("training", "#F5C842"),
    "stress": ("recovery", "#D94040"), "body_battery_morning": ("recovery", "#D94040"),
    "age": ("static", "#8b949e"), "vo2max": ("static", "#8b949e"), "ftp": ("static", "#8b949e"),
    "training_experience": ("static", "#8b949e"), "gender_enc": ("static", "#8b949e"),
    "lifestyle_enc": ("static", "#8b949e"),
}
CAT_LABELS = {"sleep": "Sleep", "hrv": "HRV / HR", "training": "Training Load",
              "recovery": "Stress & Recovery", "static": "Static / Athlete"}

def _feature_color(feat):
    return FEATURE_CATEGORIES.get(feat, ("static", "#8b949e"))[1]
def _feature_cat(feat):
    return FEATURE_CATEGORIES.get(feat, ("static", "#8b949e"))[0]

def save(fig, fname):
    path = os.path.join(PLOTS_DIR, fname)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  Saved: {fname}")

print("Plot setup done.")

In [ ]:
# Plot 01 — Injury Classification
metrics = ["roc_auc", "f1_macro", "precision", "recall"]
labels  = ["ROC-AUC", "F1-Macro", "Precision", "Recall"]
models  = [("LR", raw_results["injury_clf"]["lr"]),
           ("RF", raw_results["injury_clf"]["rf"]),
           ("DNN", raw_results["injury_clf"]["mlp"])]
x = np.arange(len(metrics))
width = 0.22
offsets = np.linspace(-1, 1, 3) * width

fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
for i, (name, data) in enumerate(models):
    means = [data["mean"][m] for m in metrics]
    stds  = [data["std"][m] for m in metrics]
    bars = ax.bar(x + offsets[i], means, width*0.9, label=name, color=MODEL_COLORS[name],
                  yerr=stds, capsize=4, error_kw={"ecolor": TEXT_COL, "alpha": 0.7}, alpha=0.88, zorder=3)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean+0.005, f"{mean:.3f}",
                ha="center", va="bottom", fontsize=8, color=TEXT_COL)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.6, 1.02); ax.set_ylabel("Score")
ax.set_title("Injury Classification \u2014 Model Comparison", pad=14)
ax.legend(loc="lower right"); ax.yaxis.grid(True, zorder=0); ax.set_axisbelow(True)
fig.tight_layout()
save(fig, "01_injury_clf_comparison.png")

# Plot 02 — Load Classification
models = [("LR", raw_results["load_clf"]["lr"]),
          ("RF", raw_results["load_clf"]["rf"]),
          ("DNN", raw_results["load_clf"]["mlp"])]
fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
for i, (name, data) in enumerate(models):
    means = [data["mean"][m] for m in metrics]
    stds  = [data["std"][m] for m in metrics]
    bars = ax.bar(x + offsets[i], means, width*0.9, label=name, color=MODEL_COLORS[name],
                  yerr=stds, capsize=4, error_kw={"ecolor": TEXT_COL, "alpha": 0.7}, alpha=0.88, zorder=3)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean+0.003, f"{mean:.3f}",
                ha="center", va="bottom", fontsize=8, color=TEXT_COL)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0.85, 1.02); ax.set_ylabel("Score")
ax.set_title("Load Class Classification \u2014 Model Comparison", pad=14)
ax.legend(loc="lower right"); ax.yaxis.grid(True, zorder=0); ax.set_axisbelow(True)
fig.tight_layout()
save(fig, "02_load_clf_comparison.png")

In [ ]:
# Plot 03 — Grit Regression
models_info = [
    ("Lasso", raw_results["grit_reg"]["lasso"], ACCENT4),
    ("RF",    raw_results["grit_reg"]["rf"],    ACCENT5),
    ("DNN",   raw_results["grit_reg"]["mlp"],   ACCENT3),
]
model_names = [m[0] for m in models_info]
rmse_means = [m[1]["mean"]["rmse"] for m in models_info]
rmse_stds  = [m[1]["std"]["rmse"] for m in models_info]
mae_means  = [m[1]["mean"]["mae"] for m in models_info]
mae_stds   = [m[1]["std"]["mae"] for m in models_info]
r2_means   = [m[1]["mean"]["r2"] for m in models_info]
r2_stds    = [m[1]["std"]["r2"] for m in models_info]

x = np.arange(len(model_names))
width = 0.25
fig, ax1 = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax1.set_facecolor(PANEL_BG); ax2 = ax1.twinx(); ax2.set_facecolor(PANEL_BG)
b1 = ax1.bar(x-width, rmse_means, width*0.9, label="RMSE", color=ACCENT2, yerr=rmse_stds, capsize=4, error_kw={"ecolor": TEXT_COL, "alpha": 0.7}, alpha=0.88, zorder=3)
b2 = ax1.bar(x, mae_means, width*0.9, label="MAE", color=ACCENT1, yerr=mae_stds, capsize=4, error_kw={"ecolor": TEXT_COL, "alpha": 0.7}, alpha=0.88, zorder=3)
b3 = ax2.bar(x+width, r2_means, width*0.9, label="R\u00b2", color=ACCENT3, yerr=r2_stds, capsize=4, error_kw={"ecolor": TEXT_COL, "alpha": 0.7}, alpha=0.88, zorder=3)
ax1.set_xticks(x); ax1.set_xticklabels(model_names)
ax1.set_ylabel("RMSE / MAE", color=ACCENT2); ax2.set_ylabel("R\u00b2", color=ACCENT3)
ax1.set_ylim(0, 2.4); ax2.set_ylim(0.96, 0.995)
ax1.set_title("Grit Score Regression \u2014 Model Comparison", pad=14)
ax1.legend([b1, b2, b3], ["RMSE", "MAE", "R\u00b2"], loc="upper right")
fig.tight_layout()
save(fig, "03_grit_regression_comparison.png")

In [ ]:
# Plots 04 & 05 — Feature Importance
def plot_fi(fi_df, title, fname):
    top10 = fi_df.head(10).sort_values("Importance", ascending=True)
    colors = [_feature_color(f) for f in top10["Feature"]]
    fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
    ax.set_facecolor(PANEL_BG)
    bars = ax.barh(top10["Feature"], top10["Importance"], color=colors, alpha=0.88, height=0.65, zorder=3)
    for bar, val in zip(bars, top10["Importance"]):
        ax.text(val+0.003, bar.get_y()+bar.get_height()/2, f"{val:.4f}", va="center", fontsize=9, color=TEXT_COL)
    seen_cats = list(dict.fromkeys([_feature_cat(f) for f in top10["Feature"]]))
    cat_colors = {"sleep": "#4A90D9", "hrv": "#FFFFFF", "training": "#F5C842", "recovery": "#D94040", "static": "#8b949e"}
    handles = [mpatches.Patch(color=cat_colors[c], label=CAT_LABELS[c]) for c in seen_cats if c in cat_colors]
    ax.legend(handles=handles, loc="lower right", framealpha=0.6)
    ax.set_xlabel("Feature Importance"); ax.set_title(title, pad=14)
    ax.xaxis.grid(True, zorder=0); ax.set_axisbelow(True)
    fig.tight_layout()
    save(fig, fname)

plot_fi(tables["fi_injury"], "Feature Importances \u2014 Injury Prediction (RF)", "04_feature_importance_injury.png")
plot_fi(tables["fi_grit"], "Feature Importances \u2014 Grit Score Regression (RF)", "05_feature_importance_grit.png")

In [ ]:
# Plot 06 — Radar Chart
categories = ["AUC", "F1", "Precision", "Recall", "Accuracy"]
n = len(categories)
models_radar = {
    "LR":  [raw_results["injury_clf"]["lr"]["mean"][k] for k in ["roc_auc","f1_macro","precision","recall","accuracy"]],
    "RF":  [raw_results["injury_clf"]["rf"]["mean"][k] for k in ["roc_auc","f1_macro","precision","recall","accuracy"]],
    "DNN": [raw_results["injury_clf"]["mlp"]["mean"][k] for k in ["roc_auc","f1_macro","precision","recall","accuracy"]],
}
angles = [i/float(n)*2*pi for i in range(n)] + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
for mname, vals in models_radar.items():
    vals_plot = vals + vals[:1]
    color = MODEL_COLORS[mname]
    ax.plot(angles, vals_plot, "o-", linewidth=2, color=color, label=mname)
    ax.fill(angles, vals_plot, alpha=0.12, color=color)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(categories, size=12)
ax.set_ylim(0.6, 1.0); ax.set_yticks([0.65, 0.75, 0.85, 0.95])
ax.set_title("Injury Classification \u2014 Radar Comparison", pad=22, size=16)
ax.legend(loc="upper right", bbox_to_anchor=(1.28, 1.12))
fig.tight_layout()
save(fig, "06_model_summary_radar.png")

In [ ]:
# Plot 07 — Grit Score Distribution
gs = df_sample["grit_score"].dropna()
q25, q75 = gs.quantile(0.25), gs.quantile(0.75)
fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
ax.axvspan(gs.min()-2, q25, alpha=0.12, color=ACCENT1, label="Undertrained")
ax.axvspan(q25, q75, alpha=0.12, color=ACCENT3, label="Balanced")
ax.axvspan(q75, gs.max()+2, alpha=0.12, color=ACCENT2, label="Overreaching")
sns.histplot(gs, bins=50, kde=True, color=ACCENT4, alpha=0.65, line_kws={"linewidth": 2}, ax=ax, zorder=3)
ax.axvline(q25, color=ACCENT1, linewidth=2, linestyle="--", label=f"Q25={q25:.1f}", zorder=4)
ax.axvline(q75, color=ACCENT2, linewidth=2, linestyle="--", label=f"Q75={q75:.1f}", zorder=4)
ax.set_xlabel("Grit Score"); ax.set_ylabel("Count")
ax.set_title("Grit Score Distribution with Load Class Thresholds", pad=14)
ax.legend(loc="upper right"); fig.tight_layout()
save(fig, "07_grit_score_distribution.png")

# Plot 08 — ACWR Distribution
acwr = df_sample["acwr"].dropna()
fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
ax.axvspan(acwr.min()-0.05, 0.8, alpha=0.14, color=ACCENT2, label="Under-training (<0.8)")
ax.axvspan(0.8, 1.3, alpha=0.10, color=ACCENT3, label="Optimal (0.8-1.3)")
ax.axvspan(1.3, acwr.max()+0.05, alpha=0.14, color=ACCENT1, label="Overreaching (>1.3)")
sns.histplot(acwr, bins=50, kde=True, color=ACCENT5, alpha=0.70, line_kws={"linewidth": 2}, ax=ax, zorder=3)
ax.axvline(0.8, color=ACCENT2, linewidth=2, linestyle="--", zorder=4)
ax.axvline(1.3, color=ACCENT1, linewidth=2, linestyle="--", zorder=4)
ax.set_xlabel("ACWR"); ax.set_ylabel("Count")
ax.set_title("ACWR Distribution with Zone Thresholds", pad=14)
ax.legend(loc="upper right"); fig.tight_layout()
save(fig, "08_acwr_distribution.png")

In [ ]:
# Plot 09 — Violin Plots
features = ["deep_sleep", "sleep_quality", "rhr_trend", "hrv_zscore"]
feat_labels = ["Deep Sleep", "Sleep Quality", "RHR Trend", "HRV Z-score"]
df_plot = df_sample[features + ["injury"]].dropna()
df_plot["Injury"] = df_plot["injury"].map({0: "No Injury", 1: "Injury"})
palette = {"No Injury": "#4A90D9", "Injury": "#D94040"}

fig, axes = plt.subplots(1, 4, figsize=(16, 7), facecolor=DARK_BG)
fig.subplots_adjust(wspace=0.35)
for ax, feat, label in zip(axes, features, feat_labels):
    ax.set_facecolor(PANEL_BG)
    sns.violinplot(data=df_plot, x="Injury", y=feat, palette=palette, inner="box", ax=ax, linewidth=1.2, cut=0)
    ax.set_title(label, fontsize=13); ax.set_xlabel(""); ax.set_ylabel(label, fontsize=11)
    ax.yaxis.grid(True, zorder=0, alpha=0.5); ax.set_axisbelow(True)
fig.suptitle("Top Injury Predictors \u2014 Distribution by Injury Status", fontsize=16, y=1.02)
fig.tight_layout()
save(fig, "09_injury_vs_features_violin.png")

In [ ]:
# Plot 10 — Correlation Heatmap
available = [c for c in FEATURE_COLS if c in df_sample.columns]
corr = df_sample[available].corr()
fig, ax = plt.subplots(figsize=(14, 12), facecolor=DARK_BG)
ax.set_facecolor(PANEL_BG)
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, ax=ax, cmap=cmap, center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, linecolor=DARK_BG,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Feature Correlation Heatmap (24 Model Features)", pad=14)
fig.tight_layout()
save(fig, "10_correlation_heatmap.png")

print("\nAll 10 plots generated!")
!ls -lh /content/plots/

## 8. HP Sweep Plots (Bonus)

In [ ]:
# Plot 11 — HP Sweep: Classification (LR C, RF depth, DNN hidden)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=DARK_BG)

# LR C
ax = axes[0]; ax.set_facecolor(PANEL_BG)
df = hp_results["lr_C"]
ax.errorbar(df["C"], df["mean_auc"], yerr=df["std_auc"], marker="o", color=ACCENT1, capsize=4, linewidth=2)
ax.set_xscale("log"); ax.set_xlabel("C"); ax.set_ylabel("ROC-AUC")
ax.set_title("LR: C vs ROC-AUC", pad=10); ax.grid(True, alpha=0.3)
best_idx = df["mean_auc"].idxmax()
ax.scatter([df.loc[best_idx, "C"]], [df.loc[best_idx, "mean_auc"]], s=200, color=ACCENT3, zorder=5, marker="*")

# RF depth
ax = axes[1]; ax.set_facecolor(PANEL_BG)
df = hp_results["rf_clf_depth"]
ax.errorbar(df["max_depth"], df["mean_auc"], yerr=df["std_auc"], marker="s", color=ACCENT2, capsize=4, linewidth=2)
ax.set_xlabel("max_depth"); ax.set_ylabel("ROC-AUC")
ax.set_title("RF Clf: max_depth vs ROC-AUC", pad=10); ax.grid(True, alpha=0.3)
best_idx = df["mean_auc"].idxmax()
ax.scatter([df.loc[best_idx, "max_depth"]], [df.loc[best_idx, "mean_auc"]], s=200, color=ACCENT3, zorder=5, marker="*")

# DNN hidden
ax = axes[2]; ax.set_facecolor(PANEL_BG)
df = hp_results["dnn_clf_hidden"]
ax.errorbar(df["hidden_size"], df["mean_auc"], yerr=df["std_auc"], marker="D", color=ACCENT3, capsize=4, linewidth=2)
ax.set_xlabel("Hidden Size"); ax.set_ylabel("ROC-AUC")
ax.set_title("DNN Clf: hidden_size vs ROC-AUC", pad=10); ax.grid(True, alpha=0.3)
best_idx = df["mean_auc"].idxmax()
ax.scatter([df.loc[best_idx, "hidden_size"]], [df.loc[best_idx, "mean_auc"]], s=200, color=ACCENT2, zorder=5, marker="*")

fig.suptitle("Hyperparameter Sweep \u2014 Classification (load_class)", fontsize=16, y=1.03)
fig.tight_layout()
save(fig, "11_hp_sweep_classification.png")

# Plot 12 — HP Sweep: Regression (Lasso alpha, RF depth, DNN hidden)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=DARK_BG)

# Lasso alpha
ax = axes[0]; ax.set_facecolor(PANEL_BG)
df = hp_results["lasso_alpha"]
ax.errorbar(df["alpha"], df["mean_r2"], yerr=df["std_r2"], marker="o", color=ACCENT4, capsize=4, linewidth=2)
ax.set_xscale("log"); ax.set_xlabel("alpha"); ax.set_ylabel("R\u00b2")
ax.set_title("Lasso: alpha vs R\u00b2", pad=10); ax.grid(True, alpha=0.3)
best_idx = df["mean_r2"].idxmax()
ax.scatter([df.loc[best_idx, "alpha"]], [df.loc[best_idx, "mean_r2"]], s=200, color=ACCENT3, zorder=5, marker="*")

# RF depth
ax = axes[1]; ax.set_facecolor(PANEL_BG)
df = hp_results["rf_reg_depth"]
ax.errorbar(df["max_depth"], df["mean_r2"], yerr=df["std_r2"], marker="s", color=ACCENT2, capsize=4, linewidth=2)
ax.set_xlabel("max_depth"); ax.set_ylabel("R\u00b2")
ax.set_title("RF Reg: max_depth vs R\u00b2", pad=10); ax.grid(True, alpha=0.3)
best_idx = df["mean_r2"].idxmax()
ax.scatter([df.loc[best_idx, "max_depth"]], [df.loc[best_idx, "mean_r2"]], s=200, color=ACCENT3, zorder=5, marker="*")

# DNN hidden
ax = axes[2]; ax.set_facecolor(PANEL_BG)
df = hp_results["dnn_reg_hidden"]
ax.errorbar(df["hidden_size"], df["mean_r2"], yerr=df["std_r2"], marker="D", color=ACCENT3, capsize=4, linewidth=2)
ax.set_xlabel("Hidden Size"); ax.set_ylabel("R\u00b2")
ax.set_title("DNN Reg: hidden_size vs R\u00b2", pad=10); ax.grid(True, alpha=0.3)
best_idx = df["mean_r2"].idxmax()
ax.scatter([df.loc[best_idx, "hidden_size"]], [df.loc[best_idx, "mean_r2"]], s=200, color=ACCENT2, zorder=5, marker="*")

fig.suptitle("Hyperparameter Sweep \u2014 Regression (grit_score)", fontsize=16, y=1.03)
fig.tight_layout()
save(fig, "12_hp_sweep_regression.png")

print("\nHP sweep plots done!")

## 9. Save & Download Results

In [ ]:
# Save all results
RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ML results
def _fi_df_full(res, feature_names, top_n=15):
    fi = pd.Series(res["feature_importance"], index=feature_names)
    return fi.nlargest(top_n).reset_index().rename(columns={"index": "Feature", 0: "Importance"})

def _clf_df(res, model_labels):
    rows = []
    for key, label in model_labels:
        m = res[key]["mean"]; s = res[key]["std"]
        rows.append({
            "Model": label,
            "ROC-AUC": f"{m['roc_auc']:.3f} +/- {s['roc_auc']:.3f}",
            "F1-macro": f"{m['f1_macro']:.3f} +/- {s['f1_macro']:.3f}",
            "Precision": f"{m['precision']:.3f} +/- {s['precision']:.3f}",
            "Recall": f"{m['recall']:.3f} +/- {s['recall']:.3f}",
            "Accuracy": f"{m['accuracy']:.3f} +/- {s['accuracy']:.3f}",
        })
    return pd.DataFrame(rows)

def _reg_df(res, model_labels):
    rows = []
    for key, label in model_labels:
        m = res[key]["mean"]; s = res[key]["std"]
        rows.append({
            "Model": label,
            "RMSE": f"{m['rmse']:.3f} +/- {s['rmse']:.3f}",
            "MAE": f"{m['mae']:.3f} +/- {s['mae']:.3f}",
            "R2": f"{m['r2']:.3f} +/- {s['r2']:.3f}",
        })
    return pd.DataFrame(rows)

clf_labels = [("lr", "Logistic Regression"), ("rf", "Random Forest"), ("mlp", "DNN (MLP)")]
reg_labels = [("lasso", "Lasso + Poly"), ("rf", "Random Forest"), ("mlp", "DNN (MLP)")]

full_tables = {
    "clf_injury": _clf_df(raw_results["injury_clf"], clf_labels),
    "clf_load":   _clf_df(raw_results["load_clf"], clf_labels),
    "reg_grit":   _reg_df(raw_results["grit_reg"], reg_labels),
    "fi_injury":  _fi_df_full(raw_results["injury_clf"], feat_names),
    "fi_grit":    _fi_df_full(raw_results["grit_reg"], feat_names),
}

payload = {
    "raw": raw_results,
    "tables": full_tables,
    "feature_names": feat_names,
    "df_feat_sample": df_feat.sample(min(5000, len(df_feat)), random_state=42),
}

with open(f"{RESULTS_DIR}/ml_results.pkl", "wb") as f:
    pickle.dump(payload, f)
print("Saved ml_results.pkl")

# HP sweep results
with open(f"{RESULTS_DIR}/hp_sweep.pkl", "wb") as f:
    pickle.dump(hp_results, f)
print("Saved hp_sweep.pkl")

# Copy plots
!cp /content/plots/*.png /content/results/
print("\nAll results in /content/results/")
!ls -lh /content/results/

In [ ]:
# Zip everything for easy download
!cd /content && zip -r triml_results.zip results/ plots/

from google.colab import files
files.download("/content/triml_results.zip")
print("\nDownload started! Extract the zip and copy contents to your TriML/results/ folder.")

## Done!

After downloading `triml_results.zip`:
1. Extract it
2. Copy `ml_results.pkl` and `hp_sweep.pkl` to `TriML/results/`
3. Copy all PNG plots to `TriML/results/plots/`
4. Commit & push to GitHub

All 9 models trained, HP sweep complete, 12 plots generated.